# Pandas Importing and Exporting Data

Runnable examples using temporary files and in-memory data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO

## 1. Read and write CSV

In [2]:
df = pd.DataFrame({'name': ['Asha', 'Dev'], 'score': [80, 90]})
path = Path('/tmp/pandas_students.csv')
df.to_csv(path, index=False)
loaded = pd.read_csv(path)
print(loaded)
assert loaded.equals(df)

   name  score
0  Asha     80
1   Dev     90


## 2. Control columns, types, and missing markers

In [3]:
csv = StringIO('id;name;score\n001;Asha;80\n002;Dev;NA\n')
df = pd.read_csv(csv, sep=';', dtype={'id': 'string'}, na_values=['NA'])
print(df)
print(df.dtypes)

    id  name  score
0  001  Asha   80.0
1  002   Dev    NaN
id       string[python]
name             object
score           float64
dtype: object


## 3. Parse dates

In [4]:
csv = StringIO('name,date\nAsha,2026-01-02\nDev,2026-02-03\n')
df = pd.read_csv(csv, parse_dates=['date'])
print(df.dtypes)
print(df)

name            object
date    datetime64[ns]
dtype: object
   name       date
0  Asha 2026-01-02
1   Dev 2026-02-03


## 4. Export selected columns and missing values

In [5]:
df = pd.DataFrame({'name': ['Asha', 'Dev'], 'score': [80.0, np.nan], 'city': ['Indore', 'Pune']})
print(df.to_csv(columns=['name', 'score'], index=False, na_rep='NA'))

name,score
Asha,80.0
Dev,NA



## 5. JSON round trip

In [6]:
df = pd.DataFrame({'name': ['Asha', 'Dev'], 'score': [80, 90]})
json_text = df.to_json(orient='records', indent=2)
print(json_text)
restored = pd.read_json(StringIO(json_text), orient='records')
print(restored)

[
  {
    "name":"Asha",
    "score":80
  },
  {
    "name":"Dev",
    "score":90
  }
]
   name  score
0  Asha     80
1   Dev     90


## 6. Excel sheets

In [7]:
xlsx_path = Path('/tmp/pandas_workbook.xlsx')
with pd.ExcelWriter(xlsx_path) as writer:
    pd.DataFrame({'name': ['Asha', 'Dev']}).to_excel(writer, sheet_name='Students', index=False)
    pd.DataFrame({'subject': ['Math', 'Science']}).to_excel(writer, sheet_name='Subjects', index=False)
all_sheets = pd.read_excel(xlsx_path, sheet_name=None)
print(list(all_sheets.keys()))
print(all_sheets['Students'])

['Students', 'Subjects']
   name
0  Asha
1   Dev


## 7. Validate imported data

In [8]:
df = pd.DataFrame({'name': ['Asha', 'Dev', 'Dev'], 'score': [80.0, np.nan, np.nan]})
print('shape:', df.shape)
print('missing:\n', df.isna().sum())
print('duplicate rows:', df.duplicated().sum())

shape: (3, 2)
missing:
 name     0
score    2
dtype: int64
duplicate rows: 1


## 8. Clean and export

In [9]:
df = pd.DataFrame({'name': ['Asha', 'Asha', 'Dev'], 'score': [80.0, 80.0, np.nan]})
clean = df.drop_duplicates().copy()
clean['score'] = clean['score'].fillna(clean['score'].median())
out = Path('/tmp/pandas_clean.csv')
clean.to_csv(out, index=False)
print(pd.read_csv(out))

   name  score
0  Asha   80.0
1   Dev   80.0


## 9. Read in chunks

In [10]:
source = StringIO('value\n' + '\n'.join(map(str, range(250))))
count = sum(len(chunk) for chunk in pd.read_csv(source, chunksize=100))
print('rows processed:', count)
assert count == 250

rows processed: 250


## 10. Nested JSON

In [11]:
records = [{'name': 'Asha', 'address': {'city': 'Indore'}}, {'name': 'Dev', 'address': {'city': 'Pune'}}]
print(pd.json_normalize(records))

   name address.city
0  Asha       Indore
1   Dev         Pune
